# Retry and fall back

**The job.** Fetch some data. The fast source is flaky. Use the slow one when it
breaks, and say so.

Two separate ideas here and they are easy to confuse:

* A **fallback** is another way to do the same step. Same contract, different
  implementation. The graph does not change.
* A **branch** is a different path through the work. The graph changes shape
  depending on what happened.

Both are here. Neither is a retry loop, and that is deliberate — a retry with no
limit is how a job runs forever.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


## Two ways to get the data

`fetch.fast` fails about half the time. `fetch.slow` always works and takes
longer. Same ports, same contract, so either can fill the step.

In [2]:
import random, time as clock
rng = random.Random(3)

attempts = {"fast": 0, "slow": 0, "cache": 0}

def fetch_fast(**kw):
    attempts["fast"] += 1
    if rng.random() < 0.6:
        raise ConnectionError("the fast source timed out")
    return {"source": "fast", "rows": [1, 2, 3, 4]}

def fetch_slow(**kw):
    attempts["slow"] += 1
    clock.sleep(0.01)
    return {"source": "slow", "rows": [1, 2, 3, 4]}

def fetch_cache(**kw):
    attempts["cache"] += 1
    return {"source": "cache (yesterday)", "rows": [1, 2, 3]}

print("three ways to do one step")

three ways to do one step


In [3]:
nodes = [
    node("fetch.fast",  "fetch",  [], [("out", "Data")], runtime={"deterministic": False}),
    node("fetch.slow",  "fetch",  [], [("out", "Data")], runtime={"deterministic": False}),
    node("fetch.cache", "fetch",  [], [("out", "Data")]),
    node("grade.rows",  "grade",  [("in", "Data")], [("fresh", "Data"), ("stale", "Data")]),
    node("use.fresh",   "use",    [("in", "Data")], [("out", "Report")]),
    node("warn.stale",  "warn",   [("in", "Data")], [("out", "Report")]),
]

stages = [
    stage("fetch", "Get the data", [], [("out", "Data")], "fetch",
          ["fetch.fast", "fetch.slow", "fetch.cache"]),
    StageDefinition(id="grade", name="Fresh or stale?", kind="branch",
                    required_capabilities=("grade",),
                    inputs=(PortSpec("in", "Data"),),
                    outputs=(PortSpec("fresh", "Data"), PortSpec("stale", "Data")),
                    success="the data was graded",
                    candidates=("grade.rows",)),
    stage("use",  "Use it",       [("in", "Data")], [("out", "Report")], "use",  ["use.fresh"]),
    stage("warn", "Flag it",      [("in", "Data")], [("out", "Report")], "warn", ["warn.stale"]),
]

edges = [Edge("fetch", "grade"),
         Edge("grade", "use",  from_port="fresh"),
         Edge("grade", "warn", from_port="stale")]

bench = build("Fetch with a fallback",
              "Get the data from whichever source works, and say which one it was.",
              stages, nodes, edges)

problems: none


In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg29005321-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Get the data</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="386" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1" stroke-dasharray="6 3"/><text x="395" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Fresh or stale?</text><text x="563" y="134.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">BRANCH</text><text x="395" y="149.0" font-size="9.5" fill="#68737f">1 candidate · one way out</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Use it</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="712" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Flag it</text><text x="721" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C316.0,141.0 316.0,141.0 386,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg29005321-arrow)"/><path d="M572,141.0 C642.0,141.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg29005321-arrow)"/><text x="642.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">fresh</text><path d="M572,141.0 C642.0,141.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg29005321-arrow)"/><text x="642.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">stale</text></svg>', title='Fetch with a fallback — shape', note='3 layers, widest 2. A dashed outline takes only one way out. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

In [5]:
def grade_rows(**kw):
    """Name the port. Cached data goes down the warning path."""
    data = kw["in"]
    return ("stale", data) if "cache" in data["source"] else ("fresh", data)

def use_fresh(**kw):
    return {"status": "used", "rows": len(kw["in"]["rows"]), "source": kw["in"]["source"]}

def warn_stale(**kw):
    return {"status": "used with a warning", "rows": len(kw["in"]["rows"]),
            "source": kw["in"]["source"], "warning": "this data is not from today"}

runtime = execute.Runtime({
    "fetch.fast": fetch_fast, "fetch.slow": fetch_slow, "fetch.cache": fetch_cache,
    "grade.rows": grade_rows, "use.fresh": use_fresh, "warn.stale": warn_stale,
})

route = {"fetch": "fetch.fast", "grade": "grade.rows",
         "use": "use.fresh", "warn": "warn.stale"}
plan = compile_route(bench, route)

## Run it ten times

Same plan every time. The fast source fails at random, so the fallbacks earn
their keep on some runs and not others.

In [6]:
FALLBACKS = {"fetch": ["fetch.slow", "fetch.cache"]}

rows = []
for i in range(10):
    got = execute.run(plan, runtime, fallbacks=FALLBACKS)
    fetch_step = next(s for s in got.steps if s.stage == "fetch")
    taken = "use" if any(s.stage == "use" and not s.skipped for s in got.steps) else "warn"
    rows.append((i + 1, fetch_step.candidate, fetch_step.fell_back, taken, got.ok))

print(f"{'run':>4}  {'source used':<14}{'fell back':<11}{'path':<7}ok")
for i, source, fell, taken, ok in rows:
    print(f"{i:>4}  {source:<14}{str(fell):<11}{taken:<7}{ok}")

print(f"\nattempts: {attempts}")

 run  source used   fell back  path   ok
   1  fetch.slow    True       use    True
   2  fetch.slow    True       use    True
   3  fetch.slow    True       use    True
   4  fetch.fast    False      use    True
   5  fetch.fast    False      use    True
   6  fetch.slow    True       use    True
   7  fetch.slow    True       use    True
   8  fetch.fast    False      use    True
   9  fetch.slow    True       use    True
  10  fetch.slow    True       use    True

attempts: {'fast': 10, 'slow': 7, 'cache': 0}


The fast source was tried every single time. When it failed the slow one took
over, and the run still succeeded. Nothing in the graph changed — only which
candidate did the work, and the run says which one that was.

Here is one of those runs as a picture. The amber bar is the step that fell
back, and it is amber rather than green on purpose: a run that succeeded on its
second choice and a run that succeeded outright are not the same run, and a
green tick for both is how a source that has quietly stopped working stays
invisible for a month.

In [7]:
# Keep running until one falls back, so the picture has something to show.
fell_back_run = None
while fell_back_run is None:
    got = execute.run(plan, runtime, fallbacks=FALLBACKS)
    if any(s.fell_back for s in got.steps):
        fell_back_run = got

viz.timeline(fell_back_run, title="a run that succeeded on its second choice")

Figure(svg='<svg viewBox="0 0 1000 246" width="1000" height="246" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.011s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">fetch</text><rect x="238.2" y="77" width="612.9" height="19" rx="4" fill="#c98a2b" opacity=".78" stroke="#c98a2b" stroke-width="1"><title>fetch.slow — fell back, 10.2ms</title></rect><text x="860.2" y="91" font-size="10" fill="#68737f">10ms · fell back</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">grade</text><rect x="853.3" y="107" width="7.1" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>grade.rows — ran, 0.1ms</title></rect><text x="869.4" y="121" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">use</text><rect x="866.2" y="137" width="3.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>use.fresh — ran, 0.1ms</title></rect><text x="878.6" y="151" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">warn</text><rect x="870.0" y="167" width="2.5" height="19" rx="4" fill="#68737f" opacity=".78" stroke="#68737f" stroke-width="1"><title>warn.stale — skipped, 0.0ms — not taken — an upstream branch went the other way</title></rect><text x="881.5" y="181" font-size="10" fill="#68737f">0ms · skipped</text><rect x="190" y="214" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="223" font-size="10" fill="#68737f">failed</text> <rect x="286" y="214" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="223" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="214" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="223" font-size="10" fill="#68737f">cached</text> <rect x="478" y="214" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="223" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="214" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="223" font-size="10" fill="#68737f">ran</text></svg>', title='a run that succeeded on its second choice', note='Bars are placed at the time each step began.', width=1000, height=246)

## What happens with no fallbacks

In [8]:
bare = [execute.run(plan, runtime).ok for _ in range(10)]
print(f"without fallbacks: {sum(bare)}/10 runs succeeded")
print(f"with fallbacks:    {sum(1 for r in rows if r[4])}/10")

without fallbacks: 6/10 runs succeeded
with fallbacks:    10/10


## The branch, when the data is old

Force the cache source and the graph takes the other path.

In [9]:
forced = execute.run(compile_route(bench, dict(route, fetch="fetch.cache")), runtime)
print(forced.text())
print()
print("used path:  ", [s.stage for s in forced.steps if not s.skipped and s.stage in ("use", "warn")])
print("skipped:    ", [s.stage for s in forced.steps if s.skipped])
print("result:     ", forced.output("warn"))

plan plan:40fc266134eda159aed9f…
4 steps in 0.000s — ok
  ok   fetch            0.000s  fetch.cache
  ok   grade            0.000s  grade.rows
  skip use              0.000s  use.fresh
       not taken — an upstream branch went the other way
  ok   warn             0.000s  warn.stale

used path:   ['warn']
skipped:     ['use']
result:      {'status': 'used with a warning', 'rows': 3, 'source': 'cache (yesterday)', 'warning': 'this data is not from today'}


In [10]:
viz.timeline(forced, title="the cache path — one step skipped, not failed")

Figure(svg='<svg viewBox="0 0 1000 246" width="1000" height="246" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.000s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">fetch</text><rect x="419.3" y="77" width="197.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>fetch.cache — ran, 0.1ms</title></rect><text x="625.6" y="91" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">grade</text><rect x="648.4" y="107" width="117.6" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>grade.rows — ran, 0.1ms</title></rect><text x="775.0" y="121" font-size="10" fill="#68737f">0ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">use</text><rect x="784.8" y="137" width="2.5" height="19" rx="4" fill="#68737f" opacity=".78" stroke="#68737f" stroke-width="1"><title>use.fresh — skipped, 0.0ms — not taken — an upstream branch went the other way</title></rect><text x="796.3" y="151" font-size="10" fill="#68737f">0ms · skipped</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">warn</text><rect x="796.7" y="167" width="73.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>warn.stale — ran, 0.0ms</title></rect><text x="879.0" y="181" font-size="10" fill="#68737f">0ms · ran</text><rect x="190" y="214" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="223" font-size="10" fill="#68737f">failed</text> <rect x="286" y="214" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="223" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="214" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="223" font-size="10" fill="#68737f">cached</text> <rect x="478" y="214" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="223" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="214" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="223" font-size="10" fill="#68737f">ran</text></svg>', title='the cache path — one step skipped, not failed', note='Bars are placed at the time each step began.', width=1000, height=246)

The `use` step never ran, and it is recorded as **skipped** rather than failed —
grey in the picture, not red. A path not taken is a correct outcome. If it were
logged as a failure every branching run would look broken and nobody would read
the logs.

Three colours, three meanings, all of which finish without raising: green ran,
amber fell back to another candidate, grey was skipped by a branch. Collapsing
them into "ok" throws away the only information worth having.